# CMIP6-decadal daily `psl` concat/subset error

This notebook reproduces a failing CDS WPS workflow for daily CMIP6-decadal sea-level pressure (`psl`) data. The workflow concatenates ten EC-Earth3 realizations initialized in 1961 and selects August 1962.

The returned data are deliberately **not** opened with `resp.datasets()` so that client-side loading does not obscure the server-side behavior.


## Original WPS workflow

The JSON below is copied from the `wps:ComplexData` CDATA payload of the failing request.


In [1]:
request = {
    "inputs": {
        "psl": [
            "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r10i1p1f1.day.psl.gr.v20201216",
            "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r1i1p1f1.day.psl.gr.v20201215",
            "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r2i1p1f1.day.psl.gr.v20201215",
            "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r3i1p1f1.day.psl.gr.v20201215",
            "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r4i1p1f1.day.psl.gr.v20201216",
            "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r5i1p1f1.day.psl.gr.v20201216",
            "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r6i1p1f1.day.psl.gr.v20201216",
            "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r7i1p1f1.day.psl.gr.v20201216",
            "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r8i1p1f1.day.psl.gr.v20201216",
            "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r9i1p1f1.day.psl.gr.v20201216",
        ]
    },
    "steps": {
        "concat_psl_1": {
            "run": "concat",
            "in": {
                "collection": "inputs/psl",
                "dims": "realization",
            },
        },
        "subset_psl_1": {
            "run": "subset",
            "in": {
                "collection": "concat_psl_1/output",
                "time_components": "month:aug|year:1962",
                "time": "1962/1962",
            },
        },
    },
    "outputs": {"output": "subset_psl_1/output"},
    "doc": "workflow",
}

request


{'inputs': {'psl': ['c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r10i1p1f1.day.psl.gr.v20201216',
   'c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r1i1p1f1.day.psl.gr.v20201215',
   'c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r2i1p1f1.day.psl.gr.v20201215',
   'c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r3i1p1f1.day.psl.gr.v20201215',
   'c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r4i1p1f1.day.psl.gr.v20201216',
   'c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r5i1p1f1.day.psl.gr.v20201216',
   'c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r6i1p1f1.day.psl.gr.v20201216',
   'c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r7i1p1f1.day.psl.gr.v20201216',
   'c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r8i1p1f1.day.psl.gr.v202

## Build the equivalent Rooki workflow

Importing Rooki contacts the configured WPS service. Change `ROOK_URL` if the reproduction should run against another deployment.


In [2]:
import json
import os

os.environ["ROOK_URL"] = "http://rook.dkrz.de/wps"

from rooki import operators as ops


In [3]:
psl = ops.Input("psl", request["inputs"]["psl"])
concat = ops.Concat(psl, dims="realization")
subset = ops.Subset(
    concat,
    time=request["steps"]["subset_psl_1"]["in"]["time"],
    time_components=request["steps"]["subset_psl_1"]["in"]["time_components"],
)

serialized_request = json.loads(subset._serialise())
serialized_request


{'inputs': {'psl': ['c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r10i1p1f1.day.psl.gr.v20201216',
   'c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r1i1p1f1.day.psl.gr.v20201215',
   'c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r2i1p1f1.day.psl.gr.v20201215',
   'c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r3i1p1f1.day.psl.gr.v20201215',
   'c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r4i1p1f1.day.psl.gr.v20201216',
   'c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r5i1p1f1.day.psl.gr.v20201216',
   'c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r6i1p1f1.day.psl.gr.v20201216',
   'c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r7i1p1f1.day.psl.gr.v20201216',
   'c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r8i1p1f1.day.psl.gr.v202

## Reproduce the failure

The next cell submits the complete workflow. Run it only against the deployment being investigated.


In [4]:
from time import perf_counter

started_at = perf_counter()
resp = subset.orchestrate()
elapsed_seconds = perf_counter() - started_at

print(f"Orchestration time: {elapsed_seconds:.1f} seconds")
resp.ok, resp.status


Orchestration time: 21.7 seconds


(True, 'ProcessSucceeded')

## Inspect the response without loading data

Displaying the response preserves failure details for diagnosis. If the workflow succeeds, the final cell lists the output URLs without downloading or opening the NetCDF result.


In [5]:
resp


Metalink URL: http://rook7.cloud.dkrz.de:80/outputs/rook/c7419ea6-9be6-11f1-978e-fa163eb671ca/input.meta4, num files: 1

In [6]:
if resp.ok:
    print("Output URLs (not downloaded):")
    for url in resp.download_urls():
        print(url)


Output URLs (not downloaded):
http://rook7.cloud.dkrz.de:80/outputs/rook/d3de6c8e-9be6-11f1-90b4-fa163eb671ca/psl_day_EC-Earth3_dcppA-hindcast_r10i1p1f1_gr_19620801-19620831.nc


In [7]:
resp.size_in_mb


109.19548034667969